# Clase 4 — Archivos y excepciones: trabajar con datos reales

**Curso:** Python sin miedo — UNSL, FCFMyN · **Módulo 2, semana 4**

## Objetivos de hoy

1. **Leer y escribir archivos de texto**: la puerta de entrada a tus datos reales.
2. Procesar el formato más común de la ciencia: **CSV** (valores separados por comas).
3. Guardar y recuperar datos estructurados con **JSON**.
4. Escribir programas que no se rompen ante datos sucios: **excepciones** (`try`/`except`).

Los datos reales no viven en el código: viven en archivos que escupe el instrumento, exporta la planilla o genera la simulación. Y los datos reales vienen *sucios*: líneas vacías, valores faltantes, texto donde esperabas números. Hoy aprendemos a leerlos y a sobrevivirlos.

## 0. Preparación: generamos datos de trabajo

Ejecutá esta celda tal cual: crea los archivos de ejemplo que vamos a usar toda la clase (así no dependemos de descargas).

In [ ]:
contenido_txt = """# Registro de temperaturas - Estacion El Volcan
# Una medicion por linea, en grados Celsius
22.4
21.8
23.1

24.0
error de sensor
22.9
21.5
"""

contenido_csv = """codigo,sitio,ph,temperatura
S-01,Norte,6.8,18.2
S-02,Sur,7.4,15.1
S-03,Norte,5.9,19.0
S-04,Sur,7.1,14.8
S-05,Norte,6.5,18.7
"""

with open('temperaturas.txt', 'w', encoding='utf-8') as f:
    f.write(contenido_txt)

with open('muestras.csv', 'w', encoding='utf-8') as f:
    f.write(contenido_csv)

print('Archivos temperaturas.txt y muestras.csv creados ✔')

## 1. Leer archivos de texto

La forma correcta de abrir un archivo es con el bloque `with`: garantiza que el archivo se cierre solo, incluso si algo falla.

```python
with open('archivo.txt', encoding='utf-8') as f:
    ...  # acá el archivo está abierto como f
# acá ya se cerró automáticamente
```

In [ ]:
# Leer todo el contenido como un único string
with open('temperaturas.txt', encoding='utf-8') as f:
    contenido = f.read()

print(contenido)

In [ ]:
# Lo más útil: recorrer el archivo línea por línea
with open('temperaturas.txt', encoding='utf-8') as f:
    for linea in f:
        print(repr(linea))   # repr muestra los caracteres ocultos

Mirá la salida de `repr`: cada línea termina con `'\n'` (el salto de línea) y hay líneas vacías y comentarios. Para limpiar:

- `linea.strip()` quita espacios y saltos de línea de los bordes.
- `linea.startswith('#')` detecta comentarios.
- `float(linea)` convierte el texto a número... cuando se puede. Enseguida vemos qué pasa cuando no.

El esqueleto clásico para procesar un archivo de datos:

In [ ]:
temperaturas = []

with open('temperaturas.txt', encoding='utf-8') as f:
    for linea in f:
        linea = linea.strip()
        if linea == '' or linea.startswith('#'):
            continue                      # saltear vacías y comentarios
        temperaturas.append(linea)        # por ahora, sin convertir

print(temperaturas)
# Problema a la vista: 'error de sensor' no es un número...

## 2. Excepciones: cuando algo sale mal

Si ejecutás `float('error de sensor')`, Python lanza un **error** (en la jerga: una **excepción**) y el programa muere. Probalo:

In [ ]:
# Descomentá y ejecutá para ver el error con tus propios ojos:
# float('error de sensor')

Leer el mensaje de error **de abajo hacia arriba** es un superpoder: la última línea dice el tipo (`ValueError`) y la causa; las anteriores dicen en qué línea pasó.

### `try` / `except`: intentar y atajar

Con `try`/`except` le decimos a Python: *intentá esto, y si lanza tal error, hacé esto otro en lugar de morir*.

In [ ]:
valores = ['22.4', 'error de sensor', '23.1', 'N/A', '24.0']

limpios = []
descartados = 0
for v in valores:
    try:
        limpios.append(float(v))
    except ValueError:
        descartados += 1

print(f'Válidos: {limpios}')
print(f'Descartados: {descartados}')

### Tipos de excepciones que vas a cruzarte seguido

| Excepción | Causa típica |
|---|---|
| `ValueError` | `float('hola')` — el valor no tiene el formato esperado |
| `TypeError` | `'2' + 2` — mezclar tipos incompatibles |
| `ZeroDivisionError` | dividir por cero |
| `FileNotFoundError` | abrir un archivo que no existe |
| `KeyError` / `IndexError` | clave inexistente en dict / índice fuera de rango |

**Regla de oro:** atajá la excepción específica (`except ValueError:`), no todas (`except:`). Un `except:` pelado esconde errores que necesitás ver.

In [ ]:
# FileNotFoundError: el clásico de los nombres mal tipeados
try:
    with open('no-existo.txt', encoding='utf-8') as f:
        datos = f.read()
except FileNotFoundError:
    print('El archivo no existe: revisá el nombre y la carpeta')

### Lanzar tus propias excepciones: `raise`

Tus funciones también pueden negarse a trabajar con datos inválidos. Es mejor un error claro y temprano que un resultado silenciosamente incorrecto:

In [ ]:
def media(valores):
    """Media aritmética. Lanza ValueError si la lista está vacía."""
    if len(valores) == 0:
        raise ValueError('No se puede calcular la media de una lista vacía')
    return sum(valores) / len(valores)

print(media([1, 2, 3]))
# Descomentá para ver tu propio error en acción:
# print(media([]))

### 🖊️ Tu turno 1

Juntá todo: releé `temperaturas.txt` línea por línea y construí la lista de temperaturas **como floats**, salteando comentarios, líneas vacías y valores no numéricos (contá cuántos descartaste). Al final mostrá la cantidad de datos válidos y su media usando la función `media` de arriba.

In [ ]:
# Tu código acá


## 3. Escribir archivos

Para escribir, se abre con `'w'` (*write*, **pisa** el archivo si existe) o `'a'` (*append*, agrega al final):

In [ ]:
resultados = [('S-01', 6.8), ('S-02', 7.4), ('S-03', 5.9)]

with open('informe.txt', 'w', encoding='utf-8') as f:
    f.write('Informe de pH\n')
    f.write('=============\n')
    for codigo, ph in resultados:
        f.write(f'{codigo}: pH {ph}\n')

# Verificamos releyéndolo:
with open('informe.txt', encoding='utf-8') as f:
    print(f.read())

⚠️ `'w'` borra sin preguntar. Antes de escribir sobre un archivo de datos valioso, pensá dos veces el nombre.

## 4. CSV: el formato universal de las tablas

Un **CSV** es texto plano con una tabla: una fila por línea, columnas separadas por comas. Excel, Google Sheets, R, los loggers de campo: todos exportan CSV. Python trae el módulo `csv` en la biblioteca estándar:

In [ ]:
import csv

with open('muestras.csv', encoding='utf-8') as f:
    lector = csv.DictReader(f)        # usa la primera fila como nombres de columna
    muestras = list(lector)

print(muestras[0])                     # cada fila es un diccionario
print(muestras[0]['sitio'], muestras[0]['ph'])

Detalle importante: `csv` lee **todo como texto**. Los números hay que convertirlos:

In [ ]:
for m in muestras:
    m['ph'] = float(m['ph'])
    m['temperatura'] = float(m['temperatura'])

# Ahora sí, a trabajar: pH medio de las muestras del sitio Norte
ph_norte = [m['ph'] for m in muestras if m['sitio'] == 'Norte']
print(f'pH medio en Norte: {sum(ph_norte) / len(ph_norte):.2f}')

In [ ]:
# Escribir un CSV: DictWriter
resumen = [
    {'sitio': 'Norte', 'n': 3, 'ph_medio': 6.4},
    {'sitio': 'Sur', 'n': 2, 'ph_medio': 7.25},
]

with open('resumen.csv', 'w', encoding='utf-8', newline='') as f:
    escritor = csv.DictWriter(f, fieldnames=['sitio', 'n', 'ph_medio'])
    escritor.writeheader()
    escritor.writerows(resumen)

with open('resumen.csv', encoding='utf-8') as f:
    print(f.read())

> En el Módulo 4 vamos a leer CSV con **Pandas**, que hace esto y mucho más en una línea. Pero entender el `csv` crudo te salva cuando el archivo viene raro — y te hace entender qué hace Pandas por dentro.

## 5. JSON: guardar estructuras completas

¿Y si quiero guardar un diccionario con listas adentro, tal cual está? Para eso existe **JSON**, el formato estándar para intercambiar datos estructurados (lo usan las APIs web, los archivos de configuración... y los notebooks de Jupyter que estás leyendo). El módulo `json` convierte entre Python y texto:

In [ ]:
import json

experimento = {
    'titulo': 'Calibración del sensor de pH',
    'fecha': '2026-10-06',
    'operador': 'M. Puliti',
    'mediciones': [6.8, 7.4, 5.9, 7.1, 6.5],
    'valido': True,
}

# Guardar (dump) en un archivo
with open('experimento.json', 'w', encoding='utf-8') as f:
    json.dump(experimento, f, indent=2, ensure_ascii=False)

# Recuperar (load) desde el archivo
with open('experimento.json', encoding='utf-8') as f:
    recuperado = json.load(f)

print(recuperado['titulo'])
print(recuperado['mediciones'])
print(type(recuperado))   # volvió como diccionario, listo para usar

Regla práctica: **CSV para tablas, JSON para estructuras** (configuraciones, metadatos, resultados anidados).

### 🖊️ Tu turno 2

Guardá en `mi_experimento.json` un diccionario que describa un experimento o muestreo de **tu** área (con al menos un texto, un número, una lista y un booleano), releelo y mostrá uno de sus campos.

In [ ]:
# Tu código acá


## 6. Práctica integradora (segunda parte de la clase)

Ejecutá primero esta celda: genera `salida_simulacion.txt`, un archivo como los que produce un programa de simulación (con encabezado, datos y líneas corruptas).

In [ ]:
contenido = """# Simulacion de difusion 1D - salida v2.3
# paso tiempo concentracion
0 0.0 1.000
1 0.1 0.905
2 0.2 0.819
3 0.3 NaN
4 0.4 0.670
5 0.5 0.607
ERROR: overflow en nodo 7
6 0.6 0.549
7 0.7 0.497
8 0.8 0.449
"""

with open('salida_simulacion.txt', 'w', encoding='utf-8') as f:
    f.write(contenido)
print('salida_simulacion.txt creado ✔')

### Ejercicio 1 — Parser robusto
Leé `salida_simulacion.txt` y construí dos listas paralelas, `tiempos` y `concentraciones` (como floats), salteando comentarios, líneas de error y valores `NaN` (pista: cada línea válida tiene 3 campos — usá `linea.split()` y `try`/`except`). Informá cuántas líneas se descartaron.

In [ ]:
# Ejercicio 1


### Ejercicio 2 — Resumen a CSV
Con las listas del ejercicio 1, escribí `simulacion_limpia.csv` con columnas `tiempo,concentracion`, y agregá al final del notebook una celda que lo relea con `csv.DictReader` para verificar.

In [ ]:
# Ejercicio 2


### Ejercicio 3 — Agrupar con diccionarios
Leé `muestras.csv` (de la sección 4) y calculá la **temperatura media por sitio** usando el patrón de diccionarios acumuladores de la clase pasada. Escribí el resultado en `temperatura_por_sitio.json`.

In [ ]:
# Ejercicio 3


### Ejercicio 4 (desafío) — Bitácora con append
Escribí una función `registrar(mensaje)` que agregue al archivo `bitacora.txt` una línea con el mensaje (modo `'a'`). Usá el módulo `datetime` para anteponer fecha y hora: `from datetime import datetime` y `datetime.now()`. Llamala tres veces y mostrá el archivo resultante.

In [ ]:
# Ejercicio 4


## Resumen de la clase

| Concepto | Sintaxis clave |
|---|---|
| Leer archivo | `with open('f.txt', encoding='utf-8') as f:` + `for linea in f:` |
| Limpiar líneas | `.strip()`, `.startswith('#')`, `.split()` |
| Escribir | `open('f.txt', 'w')` (pisa) / `'a'` (agrega), `f.write(...)` |
| Atrapar errores | `try:` / `except ValueError:` — específico, nunca pelado |
| Lanzar errores | `raise ValueError('mensaje claro')` |
| CSV | `csv.DictReader(f)`, `csv.DictWriter(f, fieldnames=...)` |
| JSON | `json.dump(dato, f)`, `dato = json.load(f)` |

## Para la próxima semana

- Completá la [guía de ejercicios del módulo 2](guia-ejercicios-modulo-2.ipynb) — **entrega: viernes 16/10/2026**.
- La próxima semana empieza el Módulo 3: **programación orientada a objetos** — vamos a dejar de escribir scripts sueltos y empezar a construir nuestras propias herramientas.